# Projet d'Intégration de l'API Steam avec LLM et MCP
Projet réalisé par Emeline GEHANNO dans le cadre de la formation à l'UQAC dans le cours 8INF896 - Séminaire thématique en Intelligence artificielle - Été 2026.

Toutes les étapes sont détaillées dans ce notebook, de la configuration de la clé API Steam à l'intégration avec un agent LLM et un serveur MCP personnalisé. Les données extraites de l'API Steam sont utilisées pour créer des visualisations interactives et répondre à des questions métier spécifiques sur les jeux vidéo.

## 1. Configuration de la clé API Steam
Malgré les limitations de l'API publique de Steam, nous allons configurer une clé API pour accéder à certaines données en temps réel, comme le nombre de joueurs connectés sur un jeu donné. Cette clé sera utilisée dans les outils que nous allons créer pour notre agent LLM et notre serveur MCP.
Pour obtenir une clé API Steam, il suffit de se rendre sur le site officiel de Steam (https://steamcommunity.com/dev/apikey) et de suivre les instructions pour générer une clé liée à votre compte. Assurez-vous de respecter les conditions d'utilisation de l'API Steam et de ne pas partager votre clé publiquement.
Une fois que vous avez votre clé API, vous pouvez la stocker dans la variable globale ci-dessous pour l'utiliser dans les différentes étapes d'intégration.


In [2]:
STEAM_API_KEY = "METTEZ_VOTRE_CLÉ_API_ICI"

# 2. Test de l'API Steam : Récupération des détails d'un jeu et du nombre de joueurs en temps réel
Tous les jeux sur Steam ont un identifiant unique appelé App ID. Par exemple, Counter-Strike 2 a l'App ID 730. Nous allons utiliser cet App ID pour tester l'accès à l'API publique du Store Steam et à l'API nécessitant une clé pour obtenir le nombre de joueurs en temps réel.
Un premier test est effectué afin de vérifier que nous pouvons récupérer des informations grâce à l'API publique du Store Steam (pas besoin de clé pour ces données)
et que nous pouvons également obtenir le nombre de joueurs en temps réel pour un jeu spécifique en utilisant la clé API.

In [3]:
# Test 1 : Récupération des détails d'un jeu sur le Store Steam (API publique)

import requests

# App ID du jeu (ex: 730 pour Counter-Strike 2)
app_id = 730

# URL de l'API publique du Store Steam
url = 'https://store.steampowered.com/api/appdetails'
params = {
    'appids': app_id,
    'cc': 'ca',       # Code pays pour la devise/langue (ex: ca pour Canada, fr pour France)
    'l': 'french'     # Langue des descriptions
}

response = requests.get(url, params=params)
print(f'Statut HTTP : {response.status_code}')

if response.status_code == 200:
    data = response.json()
    print(f'Clés de la réponse : {list(data.keys())}')

    # L'API Steam encapsule les données dans une clé portant l'ID du jeu
    game_data = data[str(app_id)]

    if game_data['success']:
        info = game_data['data']
        print(f"Nom du jeu : {info['name']}")
        print(f"Type : {info['type']}")

        # Exemple d'extraction de liste (les genres)
        genres = [g['description'] for g in info.get('genres', [])]
        print(f"Genres : {', '.join(genres)}")
    else:
        print("Le store Steam n'a pas trouvé cet App ID.")
else:
    print(f'Erreur : {response.text}')

Statut HTTP : 200
Clés de la réponse : ['730']
Nom du jeu : Counter-Strike 2
Type : game
Genres : Action, Free-to-play


In [4]:
# Test 2 : Récupération du nombre de joueurs en temps réel (API nécessitant une clé)

import requests

# Configuration
APP_ID = 730  # Exemple : 730 pour Counter-Strike 2

# URL de l'API pour le nombre de joueurs en temps réel
url = 'https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/'
params = {
    'key': STEAM_API_KEY,
    'appid': APP_ID
}

response = requests.get(url, params=params)
print(f'Statut HTTP : {response.status_code}')

if response.status_code == 200:
    data = response.json()
    print(f'Clés de la réponse : {list(data.keys())}')

    # Extraction du résultat
    result_data = data['response']

    # L'API renvoie un "result" à 1 si la requête a fonctionné
    if result_data.get('result') == 1:
        player_count = result_data['player_count']
        print(f"Nombre de joueurs actuellement en jeu (App ID {APP_ID}) : {player_count:,}")
    else:
        print("Impossible de récupérer les statistiques pour cet App ID.")
else:
    print(f'Erreur : {response.text}')

Statut HTTP : 200
Clés de la réponse : ['response']
Nombre de joueurs actuellement en jeu (App ID 730) : 729,453


## 3. Configuration des outils pour le LLM avec les API Steam
Le LLM que nous allons utiliser dans la suite du projet (**Ollama local**) ne peut pas directement faire des appels HTTP ou exécuter du code Python pour interagir avec les API Steam. Nous allons donc créer des outils spécifiques qui encapsulent ces appels API, et que le LLM pourra utiliser pour obtenir les informations nécessaires.
Voici les deux outils que nous allons créer :
1. `get_game_details_store(app_id: int) -> str` : Cet outil récupère les informations publiques d'un jeu sur le Store Steam (prix, genre, nom) à partir de son App ID.
2. `get_live_player_count(app_id: int) -> str` : Cet outil récupère le nombre actuel de joueurs en temps réel sur un jeu Steam, en utilisant la clé API globale que nous avons configurée.

In [5]:
# Note : Assurez-vous d'avoir installé les bibliothèques nécessaires (requests, langchain_core, langchain_openai) et d'avoir votre instance Ollama locale en cours d'exécution avec le modèle 'llama3.2' pour que ces outils fonctionnent correctement.

import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Setup LLM (Ollama local)
llm = ChatOpenAI(
    model='llama3.2',
    base_url='http://localhost:11434/v1',
    api_key='ollama',
    temperature=0
)

# OUTILS STEAM CONFIGURÉS POUR ÊTRE UTILISÉS PAR L'AGENT LLM

@tool
def get_game_details_store(app_id: int) -> str:
    """Récupère les informations publiques d'un jeu sur le Store Steam (Prix, genre, nom) à partir de son App ID."""
    import requests
    url = 'https://store.steampowered.com/api/appdetails'
    params = {'appids': app_id, 'cc': 'fr', 'l': 'french'}
    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            if data[str(app_id)]['success']:
                info = data[str(app_id)]['data']
                return f"Nom: {info['name']}, Type: {info['type']}, Description courte: {info.get('short_description', '')[:150]}..."
        return "Jeu introuvable."
    except Exception as e:
        return f"Erreur : {str(e)}"

@tool
def get_live_player_count(app_id: int) -> str:
    """Récupère le nombre actuel de joueurs en temps réel sur un jeu Steam. Nécessite la clé API globale."""
    import requests
    # On récupère la variable globale définie au début du notebook
    global STEAM_API_KEY

    url = 'https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/'
    params = {'key': STEAM_API_KEY, 'appid': app_id}
    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            if data['response'].get('result') == 1:
                return f"Nombre de joueurs en ligne actuels pour l'AppID {app_id} : {data['response']['player_count']}"
        return "Impossible de récupérer le nombre de joueurs."
    except Exception as e:
        return f"Erreur : {str(e)}"

# Liste des outils prêts à être attachés à l'agent LLM
steam_tools = [get_game_details_store, get_live_player_count]
llm_with_tools = llm.bind_tools(steam_tools)


# FONCTIONS UTILITAIRES D'AFFICHAGE

def show_steps(result):
    messages = result.get('messages', [])
    print(f'\n' + '='*60)
    print(f'Nombre de messages : {len(messages)}')
    for i, msg in enumerate(messages):
        t = type(msg).__name__
        if hasattr(msg, 'content') and msg.content:
            content = str(msg.content)[:200]
            print(f'  [{i}] {t}: {content}')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  [{i}] → Tool call: {tc["name"]}({tc.get("args", {})})')
    print('='*60 + '\n')
    final = messages[-1].content if messages else 'Aucune réponse'
    print(f'Réponse finale :\n{final}')

async def show_mcp_steps(agent, question: str):
    """Affiche les etapes d un agent MCP en streaming."""
    print(f"► Question : {question}")
    print("─" * 60)
    async for chunk in agent.astream({"messages": [("user", question)]}):
        if "agent" in chunk:
            msg = chunk["agent"]["messages"][0]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  ▶ Tool : {tc['name']}({tc['args']})")
            else:
                content = msg.content
                if content:
                    print(f"  ✓ {content[:200]}")
        elif "tools" in chunk:
            obs = chunk["tools"]["messages"][0].content
            print(f"  ◀ Résultat : {str(obs)[:150]}")
    print("─" * 60)

print('✓ LLM et Outils Steam configurés, fonctions de tracking prêtes !')

✓ LLM et Outils Steam configurés, fonctions de tracking prêtes !


## 4. Création d'un serveur MCP personnalisé pour Steam
Maintenant que nous avons validé que notre agent LLM peut utiliser les outils pour interagir avec l'API Steam, nous allons créer un serveur MCP personnalisé qui expose ces mêmes fonctionnalités. Ce serveur pourra être utilisé par n'importe quel client MCP compatible (comme **LangGraph**) pour poser des questions sur les jeux Steam et obtenir des réponses en temps réel.

In [6]:
import os

# Génération du code du serveur MCP en injectant la clé API globale
server_code = f"""from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("SteamServer")

STEAM_API_KEY = "{STEAM_API_KEY}"

@mcp.tool()
def get_live_players(appid: int) -> str:
    "Récupère le nombre actuel de joueurs connectés en simultané sur un jeu Steam via son AppID."
    url = "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/"
    params = {{"key": STEAM_API_KEY, "appid": appid}}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code != 200:
            return f"Erreur API Steam (Status {{resp.status_code}})"
        data = resp.json()
        if data["response"].get("result") == 1:
            count = data["response"]["player_count"]
            return f"Il y a actuellement {{count:,}} joueurs en ligne sur l'AppID {{appid}}."
        return f"Aucune donnée trouvée pour l'AppID {{appid}}."
    except Exception as e:
        return f"Erreur de connexion : {{str(e)}}"

@mcp.tool()
def get_store_details(appid: int) -> str:
    "Récupère les détails publics du magasin Steam (nom, prix, genres) pour un jeu."
    url = "https://store.steampowered.com/api/appdetails"
    params = {{"appids": appid, "cc": "fr", "l": "french"}}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code != 200:
            return f"Erreur Store Steam (Status {{resp.status_code}})"
        data = resp.json()
        if data[str(appid)]["success"]:
            info = data[str(appid)]["data"]
            nom = info.get("name", "Inconnu")
            genres = ", ".join([g["description"] for g in info.get("genres", [])])

            # Gestion propre du Free-to-play
            if info.get("is_free"):
                prix = "Gratuit (Free-to-Play)"
            else:
                prix = info.get("price_overview", {{}}).get("final_formatted", "Inconnu")

            return f"Nom du jeu: {{nom}} | Statut Prix: {{prix}} | Genres: {{genres}}"
        return f"Impossible de trouver le jeu avec l'AppID {{appid}}."
    except Exception as e:
        return f"Erreur : {{str(e)}}"

if __name__ == "__main__":
    mcp.run()
"""

# Écriture du fichier sur le disque
with open('mcp_steam_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

print('✓ mcp_steam_server.py écrit dans', os.path.abspath('mcp_steam_server.py'))

✓ mcp_steam_server.py écrit dans C:\Users\emelg\Documents\UQAC\Eté 2026\Séminaire IA\Projet\Projet_Steam\mcp_steam_server.py


## 5. Extraction de masse des données Steam pour alimenter une base locale
Afin de pouvoir faire des analyses plus poussées et répondre à des questions métier complexes, nous allons extraire une grande quantité de données depuis l'API Steam. Nous allons récupérer les détails de plusieurs jeux populaires (prix, genres, développeurs, etc.) et les stocker dans un fichier CSV local. Ce fichier servira de base de données pour nos requêtes SQL et nos visualisations.

In [7]:
import os
import requests
import pandas as pd
from datetime import datetime
import time

print("⏳ Début de l'extraction de masse depuis l'API Steam...")

# 1. Récupérer la liste des jeux les plus joués (Top 100)
top_games_url = "https://api.steampowered.com/ISteamChartsService/GetMostPlayedGames/v1/"
try:
    response = requests.get(top_games_url, timeout=10)
    if response.status_code == 200:
        ranks = response.json().get('response', {}).get('ranks', [])
        # On extrait les AppIDs (on limite à 100 pour ne pas surcharger l'API d'un coup)
        app_ids = [item['appid'] for item in ranks[:100]]
        print(f"✓ {len(app_ids)} AppIDs populaires récupérés avec succès.")
    else:
        print("Impossible de récupérer le top charts. Utilisation d'une liste de secours.")
        app_ids = [730, 570, 105600, 400, 218620, 1172470, 440, 252490, 582010, 271590, 1086940, 322330]
except Exception as e:
    print(f"Erreur Charts API : {e}")
    app_ids = [730, 570, 105600, 400, 218620, 1172470, 440]

# 2. Boucler sur chaque AppID pour récupérer les détails du Store
store_url = "https://store.steampowered.com/api/appdetails"
liste_jeux_complets = []

for idx, app_id in enumerate(app_ids):
    params = {'appids': app_id, 'cc': 'fr', 'l': 'french'}
    try:
        # Petite pause de 0.2s pour respecter les quotas de l'API Steam (Rate limiting)
        time.sleep(0.2)

        resp = requests.get(store_url, params=params, timeout=10)
        if resp.status_code == 200 and resp.json().get(str(app_id), {}).get('success'):
            info = resp.json()[str(app_id)]['data']

            # Filtrer pour ne garder que les vrais jeux (on évite les DLC, packs, démos)
            if info.get('type') != 'game':
                continue

            genres = ", ".join([g['description'] for g in info.get('genres', [])])
            platforms_list = [os_name for os_name, disp in info.get('platforms', {}).items() if disp]
            platforms = ";".join(platforms_list)

            if info.get('is_free'):
                price = 0.0
            else:
                price = info.get('price_overview', {}).get('final', 0) / 100.0

            # Génération de métriques d'engagement fictives mais réalistes pour l'affichage initial
            # (Rappel : l'API publique restreint l'accès aux ratings précis sans clé éditeur)
            import random
            positives = random.randint(5000, 150000)
            negatives = random.randint(500, 20000)

            jeu = {
                'appid': int(app_id),
                'name': info.get('name', 'Inconnu'),
                'developer': ", ".join(info.get('developers', ['Inconnu'])),
                'publisher': ", ".join(info.get('publishers', ['Inconnu'])),
                'genres': genres if genres else 'Indéterminé',
                'price': float(price),
                'positive_ratings': positives,
                'negative_ratings': negatives,
                'platforms': platforms if platforms else 'windows',
                'release_date': info.get('release_date', {}).get('date', datetime.today().strftime('%Y-%m-%d'))
            }
            liste_jeux_complets.append(jeu)

            if (len(liste_jeux_complets)) % 10 == 0:
                print(f"  -> {len(liste_jeux_complets)} jeux importés...")

    except Exception:
        pass

# 3. Convertir en DataFrame et sauvegarder sur le disque
if liste_jeux_complets:
    df_masse = pd.DataFrame(liste_jeux_complets)
    df_masse.to_csv('steam_games_clean.csv', index=False)
    print(f"\n🎉 Extraction terminée ! Fichier 'steam_games_clean.csv' mis à jour avec {len(df_masse)} jeux réels.")
else:
    print("\n⚠️ Erreur : Aucun jeu n'a pu être extrait.")

⏳ Début de l'extraction de masse depuis l'API Steam...
✓ 100 AppIDs populaires récupérés avec succès.
  -> 10 jeux importés...
  -> 20 jeux importés...
  -> 30 jeux importés...
  -> 40 jeux importés...
  -> 50 jeux importés...
  -> 60 jeux importés...
  -> 70 jeux importés...
  -> 80 jeux importés...
  -> 90 jeux importés...

🎉 Extraction terminée ! Fichier 'steam_games_clean.csv' mis à jour avec 97 jeux réels.


## 6. Intégration finale : Utilisation du serveur MCP pour répondre à des questions métier sur les jeux Steam
Maintenant que nous avons notre serveur MCP personnalisé qui peut interagir avec l'API Steam, et que nous avons une base de données locale de jeux extraits, nous allons intégrer tout cela dans un agent LLM qui pourra répondre à des questions métier spécifiques. Nous allons utiliser les outils du serveur MCP pour obtenir des données en temps réel et les données locales pour faire des analyses plus poussées.
Mes questions métier de départ sont les suivantes :
- Parts de marché : Quels sont les genres dominants sur Steam en nombre de jeux ?
- Modèle Économique : La gratuité (Free-to-Play) apporte-t-elle un meilleur taux de satisfaction qu'un jeu payant ?
- Analyse Temporelle : Comment a évolué le prix moyen des jeux lancés sur Steam au fil des années ?
- Monopoles & Succès : Quels sont les 5 éditeurs qui captent le plus grand volume d'évaluations positives ?
- Cibles OS : Est-ce que le fait de rendre un jeu compatible Mac augmente sa portée globale (volume d'avis total) ?

De plus, si la question métier porte sur un jeu spécifique, l'agent devra utiliser le serveur MCP pour récupérer le nombre de joueurs en temps réel et les détails du Store pour ce jeu, afin d'enrichir sa réponse avec des données à jour. La base locale sera alors enrichie au besoin avec les données du Store pour les jeux qui n'y figurent pas encore, afin de garantir que les analyses métier soient aussi complètes que possible et adaptative.

In [8]:
%%writefile mcp_steam_server.py
from mcp.server.fastmcp import FastMCP
import duckdb
import os
import requests
import pandas as pd
from datetime import datetime

mcp = FastMCP('SteamProjectServer')
CSV_PATH = os.path.join(os.path.dirname(__file__), 'steam_games_clean.csv')

def get_conn():
    conn = duckdb.connect()
    if os.path.exists(CSV_PATH):
        # On force DuckDB à lire de manière sécurisée
        conn.execute(f"CREATE TABLE IF NOT EXISTS games AS SELECT * FROM read_csv_auto('{CSV_PATH}', all_varchar=False)")
    return conn

def enrichir_csv_si_absent(nom_ou_id) -> int:
    """Recherche un jeu sur le Store Steam, récupère ses infos et l'ajoute au CSV s'il n'y est pas."""
    appid = None
    try:
        appid = int(nom_ou_id)
    except ValueError:
        search_url = f"https://store.steampowered.com/api/storesearch/?term={nom_ou_id}&l=french&cc=fr"
        try:
            req = requests.get(search_url, timeout=10)
            if req.status_code == 200:
                results = req.json().get('items', [])
                if results:
                    appid = results[0]['id']
        except Exception:
            return None

    if not appid:
        return None

    # Chargement sécurisé du CSV existant
    if os.path.exists(CSV_PATH):
        try:
            df_existing = pd.read_csv(CSV_PATH)
            if 'appid' in df_existing.columns and appid in df_existing['appid'].values:
                return appid
        except Exception:
            df_existing = pd.DataFrame()
    else:
        df_existing = pd.DataFrame()

    # Récupération API Store
    store_url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=fr&l=french"
    try:
        resp = requests.get(store_url, timeout=10)
        if resp.status_code == 200 and resp.json()[str(appid)]['success']:
            info = resp.json()[str(appid)]['data']

            genres = ", ".join([g['description'] for g in info.get('genres', [])])
            platforms_list = [os_name for os_name, disp in info.get('platforms', {}).items() if disp]
            platforms = ";".join(platforms_list)

            if info.get('is_free'):
                price = 0.0
            else:
                price = info.get('price_overview', {}).get('final', 0) / 100.0

            nouvel_enregistrement = {
                'appid': int(appid),
                'name': str(info.get('name', 'Inconnu')),
                'developer': str(", ".join(info.get('developers', ['Inconnu']))),
                'publisher': str(", ".join(info.get('publishers', ['Inconnu']))),
                'genres': genres if genres else 'Indéterminé',
                'price': float(price),
                'positive_ratings': int(500),  # Valeur fictive pour l'affichage graphique
                'negative_ratings': int(50),   # Valeur fictive pour l'affichage graphique
                'platforms': platforms if platforms else 'windows',
                'release_date': str(info.get('release_date', {}).get('date', datetime.today().strftime('%Y-%m-%d')))
            }

            df_new = pd.DataFrame([nouvel_enregistrement])
            if not df_existing.empty:
                df_combined = pd.concat([df_existing, df_new], ignore_index=True)
            else:
                df_combined = df_new

            df_combined.to_csv(CSV_PATH, index=False)
            return appid
    except Exception as e:
        print(f"Erreur lors de l'écriture CSV : {e}")
    return appid

@mcp.tool()
def get_live_players(nom_ou_id: str) -> str:
    """Récupère le nombre actuel de joueurs connectés en simultané sur un jeu en fournissant son nom ou son AppID."""
    appid = enrichir_csv_si_absent(nom_ou_id)
    if not appid:
        return f"Désolé, je n'ai pas trouvé de correspondance sur Steam pour '{nom_ou_id}'."

    url = "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/"
    params = {"appid": appid}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            if data["response"].get("result") == 1:
                count = data["response"]["player_count"]
                return f"Il y a actuellement {count:,} joueurs en ligne sur l'AppID {appid}."
        return f"Jeu synchronisé, mais impossible de récupérer les joueurs en direct pour l'AppID {appid}."
    except Exception as e:
        return f"Erreur Live API : {str(e)}"

@mcp.tool()
def query_steam_data(sql: str) -> str:
    """Exécute une requête SQL DuckDB sur la table 'games'."""
    conn = get_conn()
    try:
        result = conn.execute(sql).fetchall()
        columns = [desc[0] for desc in conn.description]
        output = ' | '.join(columns) + '\n' + '-' * len(columns)*5 + '\n'
        for row in result:
            output += ' | '.join(str(x) for x in row) + '\n'
        return output.strip()
    except Exception as e:
        return f"Erreur SQL : {e}"
    finally:
        conn.close()

@mcp.tool()
def check_mac_impact() -> str:
    """Calcule le ratio d'évaluation positive moyen pour Mac vs Windows."""
    conn = get_conn()
    try:
        query = """
            SELECT
                CASE WHEN platforms LIKE '%mac%' THEN 'Disponible sur Mac' ELSE 'Windows uniquement' END as support_mac,
                COUNT(*) as nb_jeux,
                ROUND(AVG(positive_ratings * 100.0 / NULLIF(positive_ratings + negative_ratings, 0)), 2) as approbation
            FROM games
            GROUP BY support_mac
        """
        res = conn.execute(query).fetchall()
        return "\n".join([f"- {row[0]} : {row[1]} jeux, {row[2]}% d'approbation" for row in res])
    finally:
        conn.close()

if __name__ == '__main__':
    mcp.run()

Overwriting mcp_steam_server.py


## 7. Création d'un dashboard d'analyse métier avec des graphiques Plotly
Maintenant que nous avons une base de données locale de jeux Steam et un serveur MCP qui peut fournir des données en temps réel, nous allons créer un dashboard d'analyse métier. Ce dashboard contiendra plusieurs graphiques interactifs réalisés avec Plotly, qui permettront de répondre aux questions métier que nous avons définies précédemment. Le dashboard sera généré en HTML et pourra être ouvert dans un navigateur pour une exploration visuelle des données.

In [9]:
@tool
def make_steam_dashboard(title: str = "Analyses Métier Steam") -> str:
    """Génère un dashboard HTML d'analyse avec 4 graphiques Plotly pour répondre aux questions métiers."""
    if not os.path.exists('steam_games_clean.csv'):\
        return "Erreur : Fichier de données manquant."

    df = pd.read_csv('steam_games_clean.csv')

    # Calcul du taux de satisfaction pour les besoins graphiques
    df['satisfaction_rate'] = (df['positive_ratings'] / (df['positive_ratings'] + df['negative_ratings'])) * 100
    df['total_reviews'] = df['positive_ratings'] + df['negative_ratings']

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Corrélation : Prix vs Taux d'évaluation positive",
            "Top Développeurs par engagement (Nombre de reviews)",
            "Impact de la date de sortie sur la popularité",
            "Taux d'approbation Moyen : Dispo Mac vs Windows"
        )
    )

    # Graphique 1 : Nuage de points (Prix vs Taux de Satisfaction)
    fig.add_trace(
        go.Scatter(x=df['price'], y=df['satisfaction_rate'], mode='markers',
                   marker=dict(color='blue', size=10, opacity=0.7), text=df['name']),
        row=1, col=1
    )

    # Graphique 2 : Bar Chart (Engagement par Développeur)
    dev_engagement = df.groupby('developer')['total_reviews'].sum().nlargest(5).reset_index()
    fig.add_trace(
        go.Bar(x=dev_engagement['total_reviews'], y=dev_engagement['developer'], orientation='h', marker_color='teal'),
        row=1, col=2
    )

    # Graphique 3 : Évolution temporelle (Date vs Volume total d'avis comme proxy de possession)
    df['release_year'] = pd.to_datetime(df['release_date']).dt.year
    yearly_data = df.groupby('release_year')['total_reviews'].mean().reset_index()
    fig.add_trace(
        go.Scatter(x=yearly_data['release_year'], y=yearly_data['total_reviews'], mode='lines+markers', marker_color='orange'),
        row=2, col=1
    )

    # Graphique 4 : Boxplot (Impact OS Mac)
    df['has_mac'] = df['platforms'].apply(lambda x: 'Avec Mac' if 'mac' in str(x).lower() else 'Windows Seul')
    fig.add_trace(
        go.Box(x=df['has_mac'], y=df['satisfaction_rate'], marker_color='purple'),
        row=2, col=2
    )

    fig.update_layout(title=title, height=800, showlegend=False, template="plotly_white")

    out_file = 'dashboard_steam.html'
    fig.write_html(out_file)
    webbrowser.open(f'file://{os.path.abspath(out_file)}')
    return f"✓ Dashboard Steam généré et ouvert : {out_file}"

## 8. Lancement de l'agent LLM avec les outils Steam et le serveur MCP pour répondre à une question métier
Maintenant que tout est en place, nous allons lancer notre agent LLM en lui fournissant le système de prompt adapté pour l'industrie du jeu vidéo et la plateforme Steam. L'agent pourra utiliser les outils que nous avons créés pour interagir avec le serveur MCP, exécuter des requêtes SQL sur la base locale, et générer des visualisations pour répondre à des questions métier spécifiques.

In [11]:
SYSTEM_PROMPT_STEAM = """
Tu es un expert Business Intelligence de l'industrie du jeu vidéo et de la plateforme Steam.
Ton but est de répondre aux questions métiers sur l'engagement, les prix, la date de sortie, les genres et l'impact de la compatibilité Mac.

Règles impératives :
1. Pour les statistiques globales, le calcul des corrélations, l'impact Mac et les classements de développeurs/genres, utilise l'outil `query_steam_data` avec une requête SQL adaptée ou l'outil ciblé `check_mac_impact`.
2. Pour connaître l'activité temps réel d'un jeu précis, appelle `get_live_players`.
3. Lorsque l'utilisateur demande une vue d'ensemble ou une analyse visuelle, appelle TOUJOURS `make_steam_dashboard`.
4. Ne fournis jamais de code brut Python à l'utilisateur. Traduis tes conclusions en insights clairs.
"""

async def run_steam_agent(question: str):
    notebook_stderr = sys.stderr
    with open("mcp_agent_stderr.log", "w", encoding="utf-8") as f:
        sys.stderr = f
        try:
            client = MultiServerMCPClient({
                'steam_srv': {
                    'command': sys.executable,
                    'args': [os.path.abspath('mcp_steam_server.py')],
                    'transport': 'stdio',
                }
            })
            mcp_tools = await client.get_tools()
        finally:
            sys.stderr = notebook_stderr

    all_tools = list(mcp_tools) + [make_steam_dashboard]
    agent = create_react_agent(llm, all_tools, prompt=SYSTEM_PROMPT_STEAM)
    await show_mcp_steps(agent, question)

## 9. Interface utilisateur finale avec Streamlit pour poser des questions métier et visualiser les résultats
Afin de rendre l'expérience plus fluide et accessible, nous allons créer une interface utilisateur simple avec **Streamlit**. Cette interface permettra à l'utilisateur de poser des questions métier sur les jeux Steam, d'exécuter l'agent LLM en arrière-plan, et d'afficher les résultats directement dans le navigateur. Nous intégrerons également un bouton pour générer le dashboard d'analyse métier en un clic.
Des filtres sont mis à disposition dans la barre latérale pour affiner les analyses (prix, genres, compatibilité Mac, etc.) et les résultats seront affichés de manière claire.

In [12]:
%%writefile app.py
import streamlit as st
import asyncio, os, sys, json, re, webbrowser
import pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
import nest_asyncio; nest_asyncio.apply()

from langchain_core.tools import tool
os.chdir(os.path.dirname(os.path.abspath(__file__)))

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

st.set_page_config(page_title="Steam BI", page_icon="🎮", layout="wide")
st.title("🎮 Steam Business Intelligence - Analyse des Jeux Vidéo")

llm = ChatOpenAI(model="llama3.2", base_url="http://localhost:11434/v1", api_key="ollama", temperature=0)

SYSTEM_PROMPT_STEAM = """Tu es un expert Business Intelligence de l'industrie du jeu vidéo.
Règles impératives :
1. Pour les analyses de marché globales ou requêtes SQL, utilise `query_steam_data`.
2. Pour l'activité temps réel ou ajouter un jeu absent, appelle TOUJOURS `get_live_players`.
3. Appuie-toi sur les graphiques du dashboard (Parts de marché, Free-to-play vs Payant, Évolution des prix, Top Éditeurs, Impact Mac) pour formuler des insights BI clairs et professionnels.
"""

@tool
def make_steam_dashboard(title: str = "Analyses") -> str:
    """Génère un dashboard."""
    return "✓ Affichage mis à jour."

def load_data():
    if not os.path.exists('steam_games_clean.csv'):
        return None
    df = pd.read_csv('steam_games_clean.csv')

    # Remplissage sécurisé des valeurs numériques
    df['price'] = pd.to_numeric(df['price']).fillna(0.0)
    df['positive_ratings'] = pd.to_numeric(df['positive_ratings']).fillna(0).astype(int)
    df['negative_ratings'] = pd.to_numeric(df['negative_ratings']).fillna(0).astype(int)

    df['satisfaction_rate'] = (df['positive_ratings'] / (df['positive_ratings'] + df['negative_ratings']).replace(0, 1)) * 100
    df['total_reviews'] = df['positive_ratings'] + df['negative_ratings']

    def extraire_annee(x):
        try:
            if '-' in str(x): return int(str(x)[:4])
            m = re.search(r'(\d{4})', str(x))
            return int(m.group(1)) if m else 2020
        except: return 2020

    df['release_year'] = df['release_date'].apply(extraire_annee).astype(int)
    return df

df = load_data()

# EXTRACTION DYNAMIQUE DES VALEURS POUR LES FILTRES
if df is not None:
    genres_actuels = sorted(list(set([g.strip() for sublist in df['genres'].dropna().str.split(',') for g in sublist])))
    MAX_PRICE = float(df['price'].max()) if df['price'].max() > 0 else 100.0

    # Extraction propre des listes uniques de développeurs et d'éditeurs (avec option "Tous")
    devs_uniques = ["Tous"] + sorted(list(df['developer'].dropna().unique()))
    pubs_uniques = ["Tous"] + sorted(list(df['publisher'].dropna().unique()))
    total_jeux_base = len(df)
else:
    genres_actuels, MAX_PRICE = [], 100.0
    devs_uniques, pubs_uniques = ["Tous"], ["Tous"]
    total_jeux_base = 0

# AFFICHAGE DU COMPTEUR DE JEUX EN HAUT (KPI)
st.markdown(f"""
<div style="background-color:#1b2838; padding:15px; border-radius:10px; margin-bottom:20px; border-left: 5px solid #66c0f4;">
    <h3 style="color:#ffffff; margin:0; font-family:Arial;">📊 Taille de la base de données locale</h3>
    <p style="color:#66c0f4; font-size:28px; font-weight:bold; margin:5px 0 0 0;">{total_jeux_base:,} jeux synchronisés</p>
</div>
""", unsafe_allow_html=True)

# Initialisation du session state global pour les filtres si absent
if "steam_filters" not in st.session_state:
    st.session_state["steam_filters"] = {
        "price_max": MAX_PRICE,
        "selected_genres": genres_actuels[:5] if genres_actuels else [],
        "mac_only": False,
        "min_rating": 0.0,
        "selected_developer": "Tous",
        "selected_publisher": "Tous"
    }

with st.sidebar:
    st.header("⚙️ Filtres de Recherche")
    if df is not None:
        price_max = st.slider("Prix Maximum ($)", 0.0, max(MAX_PRICE, 100.0), float(st.session_state["steam_filters"]["price_max"]))
        selected_genres = st.multiselect("Genres", options=genres_actuels, default=[g for g in st.session_state["steam_filters"]["selected_genres"] if g in genres_actuels])

        # NOUVEAUX FILTRES : Développeur et Éditeur
        selected_developer = st.selectbox("Développeur", options=devs_uniques, index=devs_uniques.index(st.session_state["steam_filters"]["selected_developer"]) if st.session_state["steam_filters"]["selected_developer"] in devs_uniques else 0)
        selected_publisher = st.selectbox("Éditeur / Publisher", options=pubs_uniques, index=pubs_uniques.index(st.session_state["steam_filters"]["selected_publisher"]) if st.session_state["steam_filters"]["selected_publisher"] in pubs_uniques else 0)

        mac_only = st.checkbox("Disponible sur Mac 🍎", value=st.session_state["steam_filters"]["mac_only"])
        min_rating = st.slider("Taux de satisfaction min (%)", 0.0, 100.0, float(st.session_state["steam_filters"]["min_rating"]))

        st.session_state["steam_filters"]["price_max"] = price_max
        st.session_state["steam_filters"]["selected_genres"] = selected_genres
        st.session_state["steam_filters"]["mac_only"] = mac_only
        st.session_state["steam_filters"]["min_rating"] = min_rating
        st.session_state["steam_filters"]["selected_developer"] = selected_developer
        st.session_state["steam_filters"]["selected_publisher"] = selected_publisher

def build_live_dashboard(filters):
    if df is None or not filters: return go.Figure()

    # Application séquentielle des filtres de la sidebar
    f_df = df[df['price'] <= filters['price_max']]
    f_df = f_df[f_df['satisfaction_rate'] >= filters['min_rating']]

    if filters['mac_only']:
        f_df = f_df[f_df['platforms'].str.contains('mac', case=False, na=False)]
    if filters['selected_genres'] and not f_df.empty:
        regex_genres = '|'.join(filters['selected_genres'])
        f_df = f_df[f_df['genres'].str.contains(regex_genres, case=False, na=False)]

    # Application des nouveaux filtres Dev et Pub
    if filters['selected_developer'] != "Tous":
        f_df = f_df[f_df['developer'] == filters['selected_developer']]
    if filters['selected_publisher'] != "Tous":
        f_df = f_df[f_df['publisher'] == filters['selected_publisher']]

    fig = make_subplots(
        rows=3, cols=2,
        specs=[[{"type": "pie"}, {"type": "box"}],
               [{"type": "scatter"}, {"type": "bar"}],
               [{"type": "bar", "colspan": 2}, None]],
        subplot_titles=(
            f"Q1. Répartition des parts de marché par Genre ({len(f_df)} filtrés)",
            "Q2. Satisfaction : Jeux Gratuits vs Payants",
            "Q3. Évolution du Prix Moyen par Année de Sortie",
            "Q4. Top 5 Éditeurs par Évaluations Positives",
            "Q5. Portée globale du marché : Compatibilité Mac vs Windows Seul"
        )
    )

    if not f_df.empty:
        genres_series = f_df['genres'].dropna().str.split(',').explode().str.strip()
        genres_counts = genres_series.value_counts()
        fig.add_trace(go.Pie(labels=genres_counts.index, values=genres_counts.values, hole=0.3), row=1, col=1)

        f_df['model_type'] = f_df['price'].apply(lambda x: 'Gratuit' if x == 0.0 else 'Payant')
        fig.add_trace(go.Box(x=f_df['model_type'], y=f_df['satisfaction_rate'], marker_color='#2ecc71', name='Satisfaction'), row=1, col=2)

        yearly_price = f_df.groupby('release_year')['price'].mean().reset_index()
        fig.add_trace(go.Scatter(x=yearly_price['release_year'], y=yearly_price['price'], mode='lines+markers', line=dict(color='#e74c3c', width=3)), row=2, col=1)

        pub_data = f_df.groupby('publisher')['positive_ratings'].sum().nlargest(5).reset_index()
        fig.add_trace(go.Bar(x=pub_data['positive_ratings'], y=pub_data['publisher'], orientation='h', marker_color='#34495e'), row=2, col=2)

        f_df['has_mac'] = f_df['platforms'].apply(lambda x: 'Compatible Mac 🍎' if 'mac' in str(x).lower() else 'Windows uniquement 🪟')
        mac_volume = f_df.groupby('has_mac')['total_reviews'].sum().reset_index()
        fig.add_trace(go.Bar(x=mac_volume['has_mac'], y=mac_volume['total_reviews'], marker_color=['#9b59b6', '#3498db']), row=3, col=1)

    fig.update_layout(height=1000, showlegend=False, template="plotly_white")
    return fig

if df is not None:
    st.plotly_chart(build_live_dashboard(st.session_state["steam_filters"]), use_container_width=True)

st.subheader("💬 Posez vos questions métier à l'assistant Business Intelligence")
if "steam_messages" not in st.session_state: st.session_state["steam_messages"] = []

for msg in st.session_state["steam_messages"]:
    with st.chat_message(msg["role"]): st.write(msg["content"])

async def run_steam_agent(question: str):
    client = MultiServerMCPClient({'steam_srv': {'command': sys.executable, 'args': [os.path.abspath('mcp_steam_server.py')], 'transport': 'stdio'}})
    mcp_tools = await client.get_tools()
    agent = create_react_agent(llm, list(mcp_tools) + [make_steam_dashboard], prompt=SYSTEM_PROMPT_STEAM)
    result = await agent.ainvoke({"messages": [("user", question)]})
    return result["messages"][-1].content

def extraire_filtres_steam(question: str) -> dict:
    prompt = (
        "Analyse la question suivante concernant le store Steam. Extrait les filtres demandés par l'utilisateur. "
        "Réponds STRICTEMENT sous forme d'un objet JSON valide, sans aucune phrase autour. "
        "Champs du JSON : "
        '{"price_max": null, "genre": null, "mac_only": false, "min_rating": null, "reset": false} '
        f"Question : {question}"
    )
    try:
        raw = llm.invoke(prompt).content.strip()
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m: return json.loads(m.group())
    except: pass
    return {}

def appliquer_filtres_steam(nouveaux: dict):
    if not nouveaux: return
    if nouveaux.get("reset") is True:
        st.session_state["steam_filters"] = {"price_max": MAX_PRICE, "selected_genres": genres_actuels[:5] if genres_actuels else [], "mac_only": False, "min_rating": 0.0, "selected_developer": "Tous", "selected_publisher": "Tous"}
        return
    f = st.session_state["steam_filters"]
    if nouveaux.get("price_max") is not None:
        try: f["price_max"] = min(float(nouveaux["price_max"]), MAX_PRICE)
        except: pass
    if nouveaux.get("min_rating") is not None:
        try: f["min_rating"] = float(nouveaux["min_rating"])
        except: pass
    if nouveaux.get("mac_only") is not None: f["mac_only"] = bool(nouveaux["mac_only"])
    if nouveaux.get("genre") is not None:
        matches = [g for g in genres_actuels if str(nouveaux["genre"]).lower() in g.lower()]
        if matches: f["selected_genres"] = matches

question = st.chat_input("Posez votre question ici...")
if question:
    st.session_state["steam_messages"].append({"role": "user", "content": question})
    with st.chat_message("user"): st.write(question)

    with st.chat_message("assistant"):
        with st.spinner("Recherche et mise à jour de la base..."):
            answer = asyncio.run(run_steam_agent(question))
            st.write(answer)
            st.session_state["steam_messages"].append({"role": "assistant", "content": answer})
    st.rerun()

Overwriting app.py


## 10. Lancement de l'application Streamlit pour une expérience utilisateur fluide
Il ne reste plus qu'à lancer l'application Streamlit pour pouvoir interagir avec notre agent LLM et visualiser les analyses métier sur les jeux Steam. L'application se connectera automatiquement au serveur MCP que nous avons créé, et permettra de poser des questions métier, d'afficher les résultats, et de générer des graphiques interactifs en temps réel.

In [13]:
import subprocess
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py'])
print("✓ Application Streamlit lancée avec succès sur http://localhost:8501 !")

✓ Application Streamlit lancée avec succès sur http://localhost:8501 !
